# Unit 4 practical: prompting a real small model
> **Status: written but NOT executed** while the course was authored (the authoring machine had no GPU, no model and no API key).
> Run it on Google Colab (Runtime > Change runtime type > T4 GPU is enough, CPU also works but is slow). If a cell fails, the model name or a library
> version has probably changed: the *ideas* and the helper code from the course zip are what matter.

You will: (1) load a small instruction-tuned model, (2) compare zero-shot and few-shot prompts, (3) get validated JSON out of it, (4) run a tool-calling loop,
(5) compare two prompts with a paired bootstrap interval.

In [ ]:
!pip -q install transformers accelerate
!wget -q https://krunalpanchal1995.github.io/banu.github.io/downloads/u4/u4-code.zip -O u4.zip && unzip -q -o u4.zip
import sys; sys.path.insert(0, "u4-code/codes/llm/prompting_tools")
import prompting_tools as P

## 1. Load a small chat model
`Qwen/Qwen2.5-0.5B-Instruct` exists on the Hugging Face hub (checked while writing). Any small chat model works.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
name = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, torch_dtype="auto", device_map="auto")

class HFLLM:
    """Adapter: any object with complete(messages) -> str works with prompting_tools."""
    def __init__(self, temperature=0.0, max_new_tokens=200):
        self.t, self.n = temperature, max_new_tokens
    def complete(self, messages):
        ids = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
        kw = dict(do_sample=True, temperature=self.t) if self.t > 0 else dict(do_sample=False)
        out = model.generate(ids, max_new_tokens=self.n, **kw)
        return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()

llm = HFLLM()
print(llm.complete([{"role": "user", "content": "Say hello in French."}]))

## 2. Zero-shot vs few-shot
Classify reviews. Look at the exact reply format: is it one word, a sentence, something else?

In [ ]:
system = "Classify the review as positive or negative. Reply with one word."
examples = [("I loved it, five stars", "positive"), ("Broke after a day, waste of money", "negative"), ("Fantastic value", "positive")]
q = "The battery died within a week and support ignored me."
print("zero-shot:", llm.complete([{"role": "system", "content": system}, {"role": "user", "content": q}]))
print("few-shot :", llm.complete(P.few_shot_messages(system, examples, q)))

## 3. Structured output with validation and retry

In [ ]:
schema = {"type": "object", "required": ["label", "confidence"],
          "properties": {"label": {"enum": ["positive", "negative"]}, "confidence": {"type": "number", "minimum": 0, "maximum": 1}}}
msgs = [{"role": "system", "content": 'Reply with JSON only: {"label": "positive" or "negative", "confidence": number between 0 and 1}.'},
        {"role": "user", "content": q}]
obj, attempts = P.structured_call(llm, msgs, schema, max_retries=3)
print(obj, "after", attempts, "attempt(s)")

## 4. A tool-calling loop
The small model may not follow the tool format reliably: that is itself the lesson. Watch the log.

In [ ]:
reg = P.ToolRegistry()
reg.register("calculator", lambda expression: P.safe_calc(expression),
             {"type": "object", "required": ["expression"], "properties": {"expression": {"type": "string"}}}, "evaluate arithmetic")
class ToolLLM(HFLLM):
    def complete(self, messages):
        sysmsg = {"role": "system", "content": 'To calculate, reply with ONLY this JSON: {"tool": "calculator", "arguments": {"expression": "<arithmetic>"}}. After a tool result, answer in one sentence.'}
        return super().complete([sysmsg] + [dict(m, role="user" if m["role"] == "tool" else m["role"]) for m in messages])
answer, log = P.run_tool_loop(ToolLLM(), reg, "What is 23 * (4 + 6)?")
print(answer); print(log)

## 5. Is prompt B really better than prompt A?
Forty reviews is **too few** to detect a small gain (see Section 3.3 of the PDF). Look at the interval, not just the accuracy.

In [ ]:
import numpy as np
data = [("Great product, works perfectly", 1), ("Terrible, stopped working", 0), ("Absolutely love it", 1), ("Complete waste of money", 0)] * 10
def accuracy(system_prompt):
    out = []
    for text, y in data:
        r = llm.complete([{"role": "system", "content": system_prompt}, {"role": "user", "content": text}]).lower()
        out.append(float(("positive" in r) == bool(y)))
    return np.array(out)
A = accuracy("Classify sentiment. Reply with one word: positive or negative.")
B = accuracy("You are a careful analyst. Think about the review, then reply with exactly one word: positive or negative.")
d, lo, hi = P.paired_bootstrap_ci(A, B)
print(f"A={A.mean():.2f} B={B.mean():.2f} gain={d:+.3f} 95% CI [{lo:+.3f}, {hi:+.3f}]  (n={len(A)})")

## Your turn
1. Add three more few-shot examples. Does accuracy change, and does the interval say it is real?
2. Change `temperature` to 0.8 and take a majority vote over 5 samples (`P.majority_vote_accuracy` shows when that should help).
3. Make `fit_context` drop history for a long fake conversation and check which messages survive.